In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# other required dependencies
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import string

# 1. Load data

In [3]:
# loading the dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

In [4]:
# overview of data
train.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
test.head(3)

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...


# 2. Quick sanity checks

In [6]:
train.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [7]:
test.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

# 3. Content-word length feature

In [8]:
def content_word_len(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return len(words)

def length_score(df, option_cols=OPTION_COLS):
    """Per-row min-max normalized content-word length for each option. Shape (n_rows, 5)."""
    lengths = df[option_cols].applymap(content_word_len).values.astype(float)
    row_min = lengths.min(axis=1, keepdims=True)
    row_max = lengths.max(axis=1, keepdims=True)
    span = row_max - row_min
    span[span == 0] = 1  # avoid divide-by-zero when all 5 options tie in length
    return (lengths - row_min) / span

# 4. Build the TF-IDF vectorizer

In [9]:
def combined_text(df):
    return df[['prompt'] + OPTION_COLS].astype(str).agg(' '.join, axis=1)

all_text = pd.concat([combined_text(train), combined_text(test)], ignore_index=True)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(all_text)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Vocabulary size: 2762


# 5. Cosine similarity feature

In [10]:
def cosine_scores(df, vectorizer, option_cols=OPTION_COLS):
    """Per-row cosine similarity between prompt and each option. Shape (n_rows, 5)."""
    prompt_vecs = vectorizer.transform(df['prompt'].astype(str))
    sims = {}
    for opt in option_cols:
        opt_vecs = vectorizer.transform(df[opt].astype(str))
        sims[opt] = np.array(prompt_vecs.multiply(opt_vecs).sum(axis=1)).flatten()
    return np.column_stack([sims[opt] for opt in option_cols])

# 6. Blend: 80% length / 20% cosine

In [30]:
LENGTH_WEIGHT = 0.8
COSINE_WEIGHT = 0.2

In [31]:
def blended_rank(len_scores, cos_scores, option_cols=OPTION_COLS):
    combined = LENGTH_WEIGHT * len_scores + COSINE_WEIGHT * cos_scores
    ranked_idx = np.argsort(-combined, axis=1)
    return np.array(option_cols)[ranked_idx]

In [ ]:
true_answers = train['answer'].values

# 7. Predict on train to validate

In [12]:
train_len_scores = length_score(train)
train_cos_scores = cosine_scores(train, vectorizer)

/tmp/ipykernel_287/669519934.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lengths = df[option_cols].applymap(content_word_len).values.astype(float)


In [32]:
train_ranked = blended_rank(train_len_scores, train_cos_scores)

rank_position = np.array([
    list(row).index(ans) for row, ans in zip(train_ranked, true_answers)
])

pd.Series(rank_position).value_counts(normalize=True).sort_index().round(3)

0    0.430
1    0.196
2    0.164
3    0.111
4    0.098
Name: proportion, dtype: float64

# 8. Validate

In [36]:
train_top3 = train_ranked[:, :3]

check = train[['id', 'prompt', 'answer']].copy()
check['prediction_top3'] = [' '.join(row) for row in train_top3]
check['top1_correct'] = [pred[0] == ans for pred, ans in zip(train_top3, train['answer'])]
check['correct_in_top3'] = [ans in pred for pred, ans in zip(train_top3, train['answer'])]

check.head(15)

,id,prompt,answer,prediction_top3,top1_correct,correct_in_top3
0,1,Pick the best possible answer: What is Martin ...,B,A B C,False,True
1,2,What is accelerator-based light-ion fusion?,A,A D E,True,True
2,3,Determine the correct option: What is the term...,C,A B C,False,True
3,4,Select the most accurate option: What is Marti...,B,A B C,False,True
4,5,Identify the correct statement: What is the co...,A,A B C,True,True
5,6,Identify the correct statement: What is the ef...,C,A B C,False,True
6,7,"Which of the following is correct? What is a ""...",E,B E D,False,True
7,8,Select the most accurate option: What is the L...,A,A B E,True,True
8,9,Select the most accurate option: What are the ...,A,A B D,True,True
9,10,Which of the following is correct? What is the...,A,A B D,True,True


In [37]:
n = len(check)
print(f"Top-1 exact match:      {check['top1_correct'].sum()} / {n}  ({check['top1_correct'].mean():.3f})")
print(f"Correct in top-3:       {check['correct_in_top3'].sum()} / {n}  ({check['correct_in_top3'].mean():.3f})")
print(f"Wrong (not in top-3):   {(~check['correct_in_top3']).sum()} / {n}  ({(~check['correct_in_top3']).mean():.3f})")

Top-1 exact match:      859 / 2000  (0.429)
Correct in top-3:       1581 / 2000  (0.790)
Wrong (not in top-3):   419 / 2000  (0.209)


# 9. Predict on test dataset

In [38]:
test_len_scores = length_score(test)
test_cos_scores = cosine_scores(test, vectorizer)

test_ranked = blended_rank(test_len_scores, test_cos_scores)
test_top3 = test_ranked[:, :3]

predictions = [' '.join(row) for row in test_top3]
test_top3[:5]

/tmp/ipykernel_287/669519934.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lengths = df[option_cols].applymap(content_word_len).values.astype(float)


array([['B', 'C', 'D'],
       ['A', 'B', 'C'],
       ['E', 'C', 'B'],
       ['E', 'C', 'A'],
       ['A', 'C', 'B']], dtype='<U1')

# 10. Build the submission file

In [39]:
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': predictions
})

submission.to_csv('submission.csv', index=False)
submission.shape

(500, 2)